# KNNFewShot

Retrieves the nearest labeled examples for each input, then builds a few-shot program on demand.

This notebook is self-contained and targets the stable DSPy 3.x API pinned by
`requirements.txt`. Its default data and optimizer budgets are smoke-test sized;
set `TRAIN_LIMIT=0`, `VAL_LIMIT=0`, and `EVAL_LIMIT=0` for the complete split.

In [ ]:
%pip install -r ../requirements.txt -q sentence-transformers

In [ ]:
import os
import random
from pathlib import Path

import dspy
import pandas as pd
from dotenv import load_dotenv

# Resolve paths whether Jupyter starts in the repo root or chapter06/.
cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "data" / "ai_vs_human200.csv").exists() else cwd.parent
DATA_PATH = REPO_ROOT / "data" / "ai_vs_human200.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError("Start Jupyter in the repo root or chapter06 directory.")

load_dotenv(REPO_ROOT / ".env")
if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError("Set OPENAI_API_KEY in the repo's .env file before running this notebook.")

# Chapter 6 defaults: Luna for high-volume task calls, Sol for reflection.
# Override either slug without editing the notebook.
TASK_MODEL = os.getenv("TASK_MODEL", "openai/gpt-5.6-luna")
REFLECTION_MODEL = os.getenv("REFLECTION_MODEL", "openai/gpt-5.6-sol")
NUM_THREADS = int(os.getenv("DSPY_NUM_THREADS", "4"))
TRAIN_LIMIT = int(os.getenv("TRAIN_LIMIT", "32"))
VAL_LIMIT = int(os.getenv("VAL_LIMIT", "12"))
EVAL_LIMIT = int(os.getenv("EVAL_LIMIT", "12"))

# DSPy 3.2 treats GPT-5-family models as reasoning models and rejects small
# max_tokens caps. Leave the provider default in place rather than setting an
# invalid value below 16,000.
task_lm = dspy.LM(TASK_MODEL)
reflection_lm = dspy.LM(REFLECTION_MODEL)
dspy.configure(lm=task_lm)

print(f"DSPy {dspy.__version__}")
print(f"Task model: {TASK_MODEL}")
print(f"Reflection model: {REFLECTION_MODEL}")

In [ ]:
class DetectAIText(dspy.Signature):
    """Decide whether the supplied passage was generated by an AI."""

    text: str = dspy.InputField(desc="Passage to classify")
    is_ai: bool = dspy.OutputField(desc="True for AI-generated text; otherwise False")


class AIDetector(dspy.Module):
    def __init__(self):
        super().__init__()
        self.detect = dspy.ChainOfThought(DetectAIText)

    def forward(self, text: str):
        return self.detect(text=text)


def as_bool(value) -> bool:
    if isinstance(value, bool):
        return value
    normalized = str(value).strip().lower()
    if normalized in {"true", "1", "yes"}:
        return True
    if normalized in {"false", "0", "no"}:
        return False
    raise ValueError(f"Cannot interpret {value!r} as a boolean")


def exact_match(example, prediction, trace=None) -> float:
    return float(as_bool(prediction.is_ai) == bool(example.is_ai))


def feedback_metric(example, prediction, trace=None, **kwargs):
    score = exact_match(example, prediction)
    feedback = (
        "The classification is correct. Preserve the reasoning cues that led to it."
        if score
        else f"The classification is incorrect. The expected is_ai value is {bool(example.is_ai)}. "
             "Focus on concrete stylistic evidence instead of topic or polish alone."
    )
    return dspy.Prediction(score=score, feedback=feedback)


frame = pd.read_csv(DATA_PATH)
examples = [
    dspy.Example(text=row.text, is_ai=bool(row.is_ai)).with_inputs("text")
    for row in frame.itertuples(index=False)
]
random.Random(42).shuffle(examples)

# Deterministic 50/25/25 split; environment variables keep default runs inexpensive.
train_end = len(examples) // 2
val_end = train_end + len(examples) // 4
full_trainset, full_valset, full_testset = (
    examples[:train_end], examples[train_end:val_end], examples[val_end:]
)
trainset = full_trainset[:TRAIN_LIMIT] if TRAIN_LIMIT else full_trainset
valset = full_valset[:VAL_LIMIT] if VAL_LIMIT else full_valset
testset = full_testset[:EVAL_LIMIT] if EVAL_LIMIT else full_testset

def evaluate(program, dataset=testset):
    runner = dspy.Evaluate(
        devset=dataset,
        metric=exact_match,
        num_threads=NUM_THREADS,
        max_errors=max(1, len(dataset)),
        provide_traceback=True,
        failure_score=0.0,
        display_progress=True,
        display_table=5,
    )
    result = runner(program)
    failed = [index for index, (_, prediction, _) in enumerate(result.results) if not hasattr(prediction, "is_ai")]
    if failed:
        raise RuntimeError(f"Evaluation failed for example indexes {failed}; inspect the tracebacks above.")
    return result.score


detector = AIDetector()
print(f"train={len(trainset)}, validation={len(valset)}, test={len(testset)}")

In [ ]:
baseline_score = evaluate(detector)
print(f"Baseline score: {baseline_score}")

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
vectorizer = dspy.Embedder(embedding_model.encode)

# DSPy 3.x API contract:
# - k, trainset, and vectorizer are constructor arguments.
# - BootstrapFewShot arguments may also be supplied here.
# - num_threads is NOT accepted: it would be forwarded to BootstrapFewShot and fail.
# - compile() receives the student, not trainset or valset.
optimizer = dspy.KNNFewShot(
    k=4,
    trainset=trainset,
    vectorizer=vectorizer,
    metric=exact_match,
    max_bootstrapped_demos=2,
    max_labeled_demos=0,
    max_rounds=1,
)
optimized_detector = optimizer.compile(detector)

In [ ]:
# KNNFewShot retrieves and bootstraps examples dynamically for every input. A
# serial loop keeps any single failure visible instead of ending with the opaque
# "Execution cancelled" message produced by a parallel batch.
correct = 0
failures = []
for index, example in enumerate(testset):
    try:
        prediction = optimized_detector(**example.inputs())
        correct += int(exact_match(example, prediction))
    except Exception as exc:
        failures.append((index, type(exc).__name__, str(exc)))

if failures:
    raise RuntimeError(f"KNN evaluation failures (first shown): {failures[0]}")

optimized_score = 100.0 * correct / len(testset)
print(f"KNNFewShot score: {optimized_score:.2f}")
print(f"Baseline score: {baseline_score}")